# MicroDuck 直播 Notebook：球平衡 / FastSAC

把任务、策略、ONNX、MuJoCo 和 RDK X5/BPU 验收串成一条可复用流程。

## 0. 运行说明

建议在 `02-可运行代码/microduck-playground-stilts` 的 Python 环境中启动 Jupyter。训练模型和板端 HBM 不随 Git 提交；通过 `MICRODUCK_ONNX`、`RDK_BPU_HBM` 和 `RDK_HOST` 注入。

第一次运行请先打开 [MuJoCo 策略控制 HelloWorld](./00_MuJoCo策略控制_HelloWorld.ipynb)：自动下载行走模型，用 Python 下发前进、转向和停止命令，完整执行观测、推理、关节目标和物理更新，再进入当前任务。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。

In [1]:
from pathlib import Path
import json
import os
import platform
import shlex
import socket
import subprocess


def locate_topic_root():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / "02-可运行代码").is_dir() and (base / "01-任务资料").is_dir():
            return base
        group = base / "16-专题组队学习"
        if group.is_dir():
            matches = list(group.glob("*/02-可运行代码"))
            if matches:
                return matches[0].parent
    configured = os.getenv("MICRODUCK_TOPIC_ROOT")
    if configured and Path(configured).is_dir():
        return Path(configured).resolve()
    raise RuntimeError("找不到专题目录，请从 03-Notebook 启动 Jupyter，或设置 MICRODUCK_TOPIC_ROOT。")


TOPIC_ROOT = locate_topic_root()
PLAYGROUND_ROOT = TOPIC_ROOT / "02-可运行代码" / "microduck-playground-stilts"
OUTPUT_ROOT = TOPIC_ROOT / "03-Notebook" / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("专题目录:", TOPIC_ROOT)
print("可运行代码:", PLAYGROUND_ROOT)
print("输出目录:", OUTPUT_ROOT)
print("Jupyter 工作站:", socket.gethostname(), platform.system(), platform.machine())
print("RDK 地址:", os.getenv("RDK_HOST", "192.168.8.128"), "用户: sunrise")

专题目录: /home/nvidia/microduck-course-run/topic
可运行代码: /home/nvidia/microduck-course-run/topic/02-可运行代码/microduck-playground-stilts
输出目录: /home/nvidia/microduck-course-run/topic/03-Notebook/outputs
Jupyter 工作站: spark-a8d2 Linux aarch64
RDK 地址: 192.168.8.128 用户: sunrise


## 1. 模型契约与本地检查

目标契约是 actor `obs[1, 54] -> action[1, 14]`。如果本任务没有随专题提交 checkpoint/ONNX，Notebook 会明确显示“模型未提供”，不会用别的任务模型冒充当前任务结果。

In [2]:
TASK_ID = 'microduck-ball-balance'
MODEL_HINT = None
import sys
sys.path.insert(0, str(TOPIC_ROOT / "03-Notebook"))
from notebook_runtime import resolve_model, policy_python, inspect_policy, local_model_check, verify_gpu_report, verify_bpu_report
TASK_KEY = 'ball_balance'
MODEL_PATH = resolve_model(TASK_KEY, TOPIC_ROOT, MODEL_HINT)
POLICY_PYTHON = policy_python(PLAYGROUND_ROOT)
print("任务:", TASK_KEY)
print("实际推理模型:", MODEL_PATH)
print("运动任务环境已就绪；Jupyter 与运动任务可以使用不同 Python 环境。")


任务: ball_balance
实际推理模型: /home/nvidia/microduck-course-run/converted/motrix_ball_balance_restored.opset11.onnx
运动任务环境已就绪；Jupyter 与运动任务可以使用不同 Python 环境。


In [3]:
def inspect_onnx(path):
    result = inspect_policy(path, POLICY_PYTHON)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

contract = inspect_onnx(MODEL_PATH)
print("PASS-model-contract: 已检查当前任务实际使用的模型。")

{
  "inputs": [
    [
      "obs",
      [
        "batch_size",
        54
      ]
    ]
  ],
  "outputs": [
    [
      "actions",
      [
        "batch_size",
        14
      ]
    ]
  ],
  "nodes": 48,
  "metadata": {
    "env_name": "microduck-ball-balance",
    "rllib": "motrix",
    "train_backend": "torch",
    "algo": "fastsac",
    "obs_dim": "54",
    "action_dim": "14",
    "source_checkpoint": "runs/microduck-ball-balance/motrix/torch/fastsac/26-10-21_19-01-49-352935/checkpoints/latest.pt"
  },
  "onnx_sha256": "0101d6ffdcf74e3b8ef611e70d0dd9f998c1ca54d9fd76f3f53ce90e2e7be596"
}
PASS-model-contract: 已检查当前任务实际使用的模型。


## 2. ONNX 导出入口

训练得到的是策略 checkpoint；导出时要把 actor 和观测归一化一起固化到 ONNX。下面是本任务命令模板。`--video` 会在本机录制仿真回放，新生成的视频写入 `outputs/`；本轮发布视频与运行报告保存在 `recordings/`。

```bash
python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11
```

In [4]:
EXPORT_COMMAND = "python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11" if TASK_ID == "microduck-ball-balance" else ("uv run python scripts/export.py " + (TASK_ID or "<TASK_ID>") + " --checkpoint-file <CHECKPOINT.pt> --onnx-file outputs/policy.onnx --num-envs 1 --video --video-length 250")
print(EXPORT_COMMAND)
print("导出前后检查：观测维度、动作顺序、归一化和 action clip。")

python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11
导出前后检查：观测维度、动作顺序、归一化和 action clip。


## 3. 训练路径说明：从零或从预训练 checkpoint 开始

每个 Notebook 都把训练入口和回放入口分开：可以把 `resume=False` 作为从零初始化，也可以把 `resume=True`、任务专属 checkpoint 和对应的 actor/critic/优化器状态作为预训练续训。默认演示视频使用已经验证过的任务模型，不能把它描述成当前单元刚训练出来的模型。

这里的“10 个 iteration”是 10 轮采样加策略优化，不是每个动作只训练 10 步；例如 `64 个并行环境 × 24 个控制步 × 10` 才是约 `15360` 条 transition。导航 Notebook 明确不训练，直接接入已有模型；扰动和爬梯等 Notebook 也会标明它们使用的任务专属模型或官方双模型。

## 3. FastSAC smoke 训练与 ONNX 导出

球平衡使用 MotrixLab 的 FastSAC，不复用 PPO/LSTM。默认在 Ubuntu 的 MotrixLab `.venv` 中用 64 个并行环境做 5 个 smoke iteration，并把本次 run 导出为 `outputs/motrix_ball_balance_latest.onnx`。若重新训练后要生成新的板端视频，需要同步重新执行 X5 HBM 编译单元，不能继续使用旧 HBM。

In [5]:
import shutil
import sys

MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser()
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser()
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py"
FASTSAC_ENVS = int(os.getenv("MICRODUCK_FASTSAC_ENVS", "64"))
FASTSAC_ITERATIONS = int(os.getenv("MICRODUCK_FASTSAC_ITERATIONS", "10"))
TRAIN_MODE = os.getenv("MICRODUCK_TRAIN_MODE", "pretrained")
RESTORED_CHECKPOINT = Path(os.getenv("MICRODUCK_FASTSAC_CHECKPOINT", "/home/ubuntu/workspaces/microduck_bpu_models/restored_ball_balance/checkpoints/model_0005002.pt"))
RUN_FASTSAC_SMOKE = os.getenv("MICRODUCK_RUN_FASTSAC_SMOKE", "0") == "1"
FASTSAC_ONNX = OUTPUT_ROOT / "motrix_ball_balance_latest.onnx"
print("MotrixLab:", MOTRIX_ROOT)
print("FastSAC Python:", MOTRIX_PYTHON)
print("固定输出:", FASTSAC_ONNX)

if not RUN_FASTSAC_SMOKE:
    print("已跳过 FastSAC smoke 训练：设置 MICRODUCK_RUN_FASTSAC_SMOKE=1 后重新运行。")
elif not MOTRIX_ROOT.is_dir() or not MOTRIX_PYTHON.is_file():
    print("找不到 MotrixLab 或其 Python 环境；请先在 Ubuntu 完成 MotrixLab 环境安装。")
else:
    train_cmd = [
        str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "train", str(MOTRIX_ROOT / "scripts/train.py"),
        "task=microduck-ball-balance/motrix.fastsac",
        f"num_envs={FASTSAC_ENVS}",
        "play=false", "render=false", "algo.asynchronous=false",
        "algo.device=cuda", "algo.agent.compile=false", "algo.agent.amp=false",
        # Keep the default run genuinely smoke-sized. The replay buffer must
        # be populated before FastSAC can take its first update.
        "algo.agent.learning_starts=2", "algo.agent.buffer_size=8", "algo.agent.batch_size=64",
        "algo.agent.num_updates=1", f"algo.trainer.num_learning_iterations={FASTSAC_ITERATIONS}",
    ]
    if TRAIN_MODE == "pretrained":
        probe = subprocess.run([str(MOTRIX_PYTHON), "-c", "import torch,sys; print(torch.load(sys.argv[1], map_location=\"cpu\", weights_only=True)[\"global_step\"])", str(RESTORED_CHECKPOINT)], check=True, capture_output=True, text=True)
        starting_step = int(probe.stdout.strip())
        train_cmd[-1] = f"algo.trainer.num_learning_iterations={starting_step + FASTSAC_ITERATIONS}"
        train_cmd.append(f"resume={RESTORED_CHECKPOINT}")
    elif TRAIN_MODE != "scratch":
        raise ValueError("MICRODUCK_TRAIN_MODE must be pretrained or scratch")
    print("开始 FastSAC smoke 训练:", " ".join(shlex.quote(x) for x in train_cmd))
    train_result = subprocess.run(train_cmd, cwd=MOTRIX_ROOT, text=True)
    if train_result.returncode != 0:
        raise RuntimeError(f"FastSAC smoke 训练失败，returncode={train_result.returncode}")
    run_root = MOTRIX_ROOT / "runs" / "microduck-ball-balance" / "motrix" / "torch" / "fastsac"
    run_dirs = sorted((p for p in run_root.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise FileNotFoundError(f"训练成功但没有找到 FastSAC run: {run_root}")
    run_dir = run_dirs[-1]
    motrix_export = Path("outputs") / "motrix_ball_balance_latest.onnx"
    export_cmd = [
        str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "export", str(MOTRIX_ROOT / "scripts/export_onnx.py"),
        f"run_dir='{run_dir}'", f"output={motrix_export.as_posix()}", "opset=11",
    ]
    print("导出 FastSAC ONNX:", " ".join(shlex.quote(x) for x in export_cmd))
    export_result = subprocess.run(export_cmd, cwd=MOTRIX_ROOT, text=True)
    exported_onnx = MOTRIX_ROOT / motrix_export
    if export_result.returncode != 0 or not exported_onnx.is_file():
        raise RuntimeError("FastSAC ONNX 导出失败。")
    shutil.copy2(exported_onnx, FASTSAC_ONNX)
    CANDIDATE_MODEL_PATH = FASTSAC_ONNX
    print("Training candidate:", CANDIDATE_MODEL_PATH, "Stable inference model:", MODEL_PATH)
    print("PASS-fastsac-model:", MODEL_PATH)
    print("实际训练 run:", run_dir)

MotrixLab: /home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab
FastSAC Python: /home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/.venv/bin/python
固定输出: /home/nvidia/microduck-course-run/topic/03-Notebook/outputs/motrix_ball_balance_latest.onnx
开始 FastSAC smoke 训练: '/home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/.venv/bin/python' '/home/nvidia/microduck-course-run/topic/02-可运行代码/microduck-playground-stilts/scripts/motrix_runner.py' train '/home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/scripts/train.py' task=microduck-ball-balance/motrix.fastsac num_envs=64 play=false render=false algo.asynchronous=false algo.device=cuda algo.agent.compile=false algo.agent.amp=false algo.agent.learning_starts=2 algo.agent.buffer_size=8 algo.agent.batch_size=64 algo.agent.num_updates=1 algo.trainer.num_learning_iterations=10


Training startup: resolving train backend for motrix/fastsac


Training startup: resolved train backend 'torch' in 0.270s (env=microduck-ball-balance, num_envs=64)
Training startup: created run context in 0.010s (run_dir=runs/microduck-ball-balance/motrix/torch/fastsac/26-10-21_19-12-51-101991)


Motphys profiler initialized: disabled


Training startup: trainer handle ready in 0.495s


...

------------------------------------------------------------------------------------
 microduck-ball-balance/motrix.fastsac - iter 0/10 (0.0%) - 0 env-steps/s - 0m00s
------------------------------------------------------------------------------------
 rollout     return       nan   ep_len     nan   episodes      0
 system      cpu  24.9% (5.0/20T, 20C)   iowait 0.0%   steal 0.0%
 system      gpu 11%  n/a
 train       warmup - filling replay buffer (0/512)
 reward(dt)  action_rate_l2 -19.8164     upright +0.0757
             ball_under_feet +0.0525     base_height +0.0299
             alive +0.0200               dof_default +0.0097
             limits_dof_pos +0.0000      undesired_contacts +0.0000
 metrics     bad_base_z +0.000   bad_dof_pos +0.000   bad_dof_vel +0.000   bad_orientation +0.000   ball_escaped +0.000   ball_feet_distance +3.462e-04   base_tilt +0.002   base_z +0.400   dof_limit_violation_max +0.000   dof_vel_abs_max +7.907
 timing      collect 89.5ms   learn 0.0ms
-----

导出 FastSAC ONNX: '/home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/.venv/bin/python' '/home/nvidia/microduck-course-run/topic/02-可运行代码/microduck-playground-stilts/scripts/motrix_runner.py' export '/home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/scripts/export_onnx.py' 'run_dir='"'"'/home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/runs/microduck-ball-balance/motrix/torch/fastsac/26-10-21_19-12-51-101991'"'"'' output=outputs/motrix_ball_balance_latest.onnx opset=11


2026-10-21 19:12:55.314329884 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


{
  "input": {
    "dtype": "float32",
    "name": "obs",
    "shape": [
      null,
      54
    ]
  },
  "max_abs_error": 2.384185791015625e-07,
  "max_rel_error": 3.173667209921405e-05,
  "output": "outputs/motrix_ball_balance_latest.onnx",
  "output_tensor": {
    "dtype": "float32",
    "name": "actions",
    "shape": [
      null,
      14
    ]
  },
  "validation_samples": 32
}


Training candidate: /home/nvidia/microduck-course-run/topic/03-Notebook/outputs/motrix_ball_balance_latest.onnx Stable inference model: /home/nvidia/microduck-course-run/converted/motrix_ball_balance_restored.opset11.onnx
PASS-fastsac-model: /home/nvidia/microduck-course-run/converted/motrix_ball_balance_restored.opset11.onnx
实际训练 run: /home/nvidia/microduck-course-run/topic/02-可运行代码/MotrixLab/runs/microduck-ball-balance/motrix/torch/fastsac/26-10-21_19-12-51-101991


## 3b. X5 HBM 编译

训练后的 ONNX 不能直接交给 RDK X5。下面固定输入输出 batch 为 1，并通过 Ubuntu 上的官方 X5 CPU Docker 工具链运行 `hb_mapper makertbin --model-type onnx`。生成的 `ball_balance.opset11.bin` 会覆盖编译目录，下一节视频使用的就是这份新 HBM。

In [6]:
COMPILE_BPU = os.getenv("MICRODUCK_COMPILE_BPU", "0") == "1"
if not COMPILE_BPU:
    print("NOT-RUN-COMPILE: 本轮使用已校准模型。")
else:
    BPU_MODEL_ROOT = Path(os.getenv("MICRODUCK_BPU_MODEL_ROOT", "/home/ubuntu/workspaces/microduck_bpu_models"))
    calibration_video = OUTPUT_ROOT / "diagnostics/calibration/ball_balance_gpu.mp4"
    calibration_video.parent.mkdir(parents=True, exist_ok=True)
    commands = [
        [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(PLAYGROUND_ROOT / "scripts/motrix_gpu_inloop_video.py"), "--onnx", str(MODEL_PATH), "--steps", "1000", "--output", str(calibration_video)],
        [str(POLICY_PYTHON), str(PLAYGROUND_ROOT / "scripts/prepare_stability_candidates.py"), "--root", str(BPU_MODEL_ROOT), "--task", "ball_balance", "--onnx", str(MODEL_PATH), "--trace", str(calibration_video.with_suffix(".trace.npz")), "--external-normalization", "--split-weights"],
        ["docker", "run", "--rm", "-v", str(BPU_MODEL_ROOT.parent) + ":/work", os.getenv("MICRODUCK_BPU_DOCKER_IMAGE", "openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8"), "hb_mapper", "makertbin", "--model-type", "onnx", "--config", "/work/" + str((BPU_MODEL_ROOT / "ball_balance_restored_int16_normalized_split_config.yaml").relative_to(BPU_MODEL_ROOT.parent))],
    ]
    for command in commands:
        subprocess.run(command, cwd=PLAYGROUND_ROOT, check=True)
    print("PASS-X5-HBM: 真实轨迹校准完成，候选模型保存到 compiled/ball_balance_restored_int16_normalized_split。")


NOT-RUN-COMPILE: 本轮使用已校准模型。


## 4. 检查 ONNX 的输入与动作输出

先确认模型的输入名称、维度和动作输出，为接入仿真准备接口。下方使用零输入做一次模型检查。
随后在本任务闭环单元中读取实时观测，重新计算动作、下发执行器并更新物理状态。
第一次接触这条流程，请先运行 [MuJoCo 策略控制 HelloWorld](./00_MuJoCo策略控制_HelloWorld.ipynb)，
亲手执行前进、转向和停止，并查看 `data.ctrl` 与 `mujoco.mj_step()` 怎样连接策略和仿真。

In [7]:
local_check = local_model_check(MODEL_PATH, POLICY_PYTHON)
print(json.dumps(local_check, ensure_ascii=False, indent=2))
print("PASS-local-onnx: 输入输出检查完成，下一单元运行任务闭环。")

{
  "providers": [
    "CPUExecutionProvider"
  ],
  "inputs": [
    [
      "obs",
      [
        "batch_size",
        54
      ]
    ]
  ],
  "outputs": [
    [
      "actions",
      [
        "batch_size",
        14
      ]
    ]
  ],
  "finite": true
}
PASS-local-onnx: 输入输出检查完成，下一单元运行任务闭环。


## 5. 本任务策略闭环与 GPU 视频

运行下方单元会启动当前任务的策略与仿真器：每个控制周期读取机器人实时观测，
调用当前任务的 actor，取得关节动作，再交给仿真环境的 `env.step(actions)` 处理执行器与物理更新。
下一周期使用更新后的状态重新推理；视频记录这一段完整运行过程。
工作站完成物理计算、渲染与编码，GPU 执行当前任务的策略网络。

脚本位于 `02-可运行代码/microduck-playground-stilts/scripts/`。
行走、扰动、高跷、篮球和摆动使用 `gpu_inloop_video.py`；球平衡使用 `motrix_gpu_inloop_video.py`；
爬梯使用 `bpu_official_ladder_pair_video.py` 的 GPU 分支与爬梯/起身双策略。
下方模型路径需要指向当前任务的权重；各任务教程说明模型下载与环境准备。

若想逐行修改速度命令并直接看到关节目标下发，先运行 [HelloWorld](./00_MuJoCo策略控制_HelloWorld.ipynb)。
RDK 推理放在下一节，接收同一控制流程中的观测并返回动作。


本次模型检查与闭环使用同一个 `MODEL_PATH`。报告记录运行编号、真实调用次数和真实观测上的 GPU/ONNX 数值对照；缺少模型或推理失败会明确报错。

## 模型训练进度与终止原因

从 ONNX 元数据找到原始检查点，恢复策略和归一化。在同一场景、同一种子下分别运行原始策略与 ONNX，对照动作误差及重置原因。

In [8]:
SOURCE_AUDIT = OUTPUT_ROOT / "diagnostics" / "motrix_checkpoint.json"
SOURCE_AUDIT.parent.mkdir(parents=True, exist_ok=True)
audit_cmd = [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video",
    str(PLAYGROUND_ROOT / "scripts/audit_motrix_checkpoint.py"),
    "--motrix-root", str(MOTRIX_ROOT), "--onnx", str(MODEL_PATH),
    "--output", str(SOURCE_AUDIT)]
audit_result = subprocess.run(audit_cmd, cwd=MOTRIX_ROOT, capture_output=True, text=True, timeout=180)
if audit_result.returncode:
    raise RuntimeError(audit_result.stderr[-3000:])
source_audit = json.loads(SOURCE_AUDIT.read_text(encoding="utf-8"))
print("模型累计训练步数:", source_audit["training_global_step"])
print("优化更新次数:", source_audit["training_update_idx"])
for backend, result in source_audit["rollouts"].items():
    print(backend, "真实控制步数:", result["steps"], "重置次数:", result["resets"],
          "终止原因:", result["termination_counts"], "最大动作误差:", result["max_abs_action_error"])
print("PASS-source-checkpoint: 原始策略与 ONNX 已在真实仿真观测上完成对照。")

模型累计训练步数: 10
优化更新次数: 8
checkpoint 真实控制步数: 250 重置次数: 17 终止原因: {'bad_orientation': 17} 最大动作误差: 3.5762786865234375e-07
onnx 真实控制步数: 250 重置次数: 17 终止原因: {'bad_orientation': 17} 最大动作误差: 3.5762786865234375e-07
PASS-source-checkpoint: 原始策略与 ONNX 已在真实仿真观测上完成对照。


In [9]:
import sys

GPU_TASK = 'ball_balance'
GPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_GPU_VIDEO_STEPS", "1500" if GPU_TASK == "basketball" else "900" if GPU_TASK == "ball_balance" else "500"))
GPU_VIDEO_OUTPUT = OUTPUT_ROOT / "gpu_videos" / f"{GPU_TASK}_gpu_latest.mp4" if GPU_TASK else None
GPU_ONNX_DEFAULTS = {
    "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/converted/basketball.opset11.onnx",
    "walking": "/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx",
    "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx",
    "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/converted/stilts25.opset11.onnx",
    "swing": "/home/ubuntu/workspaces/microduck_bpu_models/converted/swing.opset11.onnx",
    "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/converted/motrix_ball_balance_restored.opset11.onnx",
}
GPU_ONNX = MODEL_PATH
GPU_LADDER_CLIMBER = Path(os.getenv("MICRODUCK_GPU_LADDER_CLIMBER", "/home/ubuntu/workspaces/microduck_bpu_models/converted/official_ladder_climber.opset11.onnx")).expanduser()
GPU_LADDER_GETUP = Path(os.getenv("MICRODUCK_GPU_LADDER_GETUP", "/home/ubuntu/workspaces/microduck_bpu_models/converted/official_ladder_getup.opset11.onnx")).expanduser()
MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if GPU_TASK == "ball_balance" else None
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if GPU_TASK == "ball_balance" else None
GPU_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_gpu_inloop_video.py" if GPU_TASK == "ball_balance" else "gpu_inloop_video.py")
GPU_LADDER_SCRIPT = PLAYGROUND_ROOT / "scripts" / "bpu_official_ladder_pair_video.py"

if GPU_TASK == "ladder":
    gpu_ready = GPU_LADDER_SCRIPT.is_file() and GPU_LADDER_CLIMBER.is_file() and GPU_LADDER_GETUP.is_file()
    gpu_cmd = [
        str(POLICY_PYTHON), str(GPU_LADDER_SCRIPT), "--backend", "gpu",
        "--official-root", "/home/ubuntu/workspaces/microduck-playground-official/experiments/desk-climb",
        "--climber-onnx", str(GPU_LADDER_CLIMBER), "--getup-onnx", str(GPU_LADDER_GETUP),
        "--output", str(GPU_VIDEO_OUTPUT), "--seconds", "24", "--num-envs", "64", "--record-env", "17",
    ]
else:
    gpu_ready = GPU_SCRIPT.is_file() and GPU_ONNX is not None and GPU_ONNX.is_file()
    gpu_cmd = [str(POLICY_PYTHON), str(GPU_SCRIPT)] if GPU_TASK != "ball_balance" else [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(GPU_SCRIPT)]
    if GPU_TASK != "ball_balance":
        gpu_cmd += ["--task", GPU_TASK]
    gpu_cmd += ["--onnx", str(GPU_ONNX), "--output", str(GPU_VIDEO_OUTPUT), "--steps", str(GPU_VIDEO_STEPS)]

print("GPU 推理任务:", GPU_TASK)
print("GPU 模型:", GPU_ONNX if GPU_TASK != "ladder" else (GPU_LADDER_CLIMBER, GPU_LADDER_GETUP))
if not gpu_ready:
    raise FileNotFoundError("缺少任务模型、脚本或仿真环境；请按任务教程准备。")
elif GPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
    raise FileNotFoundError(MOTRIX_PYTHON)
else:
    GPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    print("开始生成 GPU 推理视频。")
    import uuid
    GPU_RUN_ID = str(uuid.uuid4())
    gpu_env = os.environ.copy()
    gpu_env["MICRODUCK_VALIDATION_RUN_ID"] = GPU_RUN_ID
    gpu_result = subprocess.run(gpu_cmd, cwd=PLAYGROUND_ROOT, env=gpu_env, capture_output=True, text=True, timeout=3600)
    safe_lines = [line for line in (gpu_result.stdout + "\n" + gpu_result.stderr).splitlines() if "GPU" in line or "PASS-" in line or "Video" in line]
    print("\n".join(safe_lines[-20:]))
    if gpu_result.returncode != 0 or not GPU_VIDEO_OUTPUT.is_file():
        log_path = OUTPUT_ROOT / "gpu_videos" / f"{GPU_TASK}_error.log"
        log_path.write_text(gpu_result.stdout + "\n" + gpu_result.stderr, encoding="utf-8")
        raise RuntimeError(f"GPU 推理失败；完整日志: {log_path}")
    GPU_REPORT = GPU_VIDEO_OUTPUT.with_suffix(".json")
    print("PASS-GPU-VIDEO:", GPU_VIDEO_OUTPUT)
    selected_models = [GPU_LADDER_CLIMBER, GPU_LADDER_GETUP] if GPU_TASK == "ladder" else [GPU_ONNX]
    gpu_report = verify_gpu_report(GPU_VIDEO_OUTPUT, GPU_RUN_ID, selected_models)
    print(gpu_report)
    print("PASS-FRESH-INFERENCE: 本次模型调用、关节动作与新视频已核对。")
    from IPython.display import Video, display
    display(Video(str(GPU_VIDEO_OUTPUT), embed=True))

GPU 推理任务: ball_balance
GPU 模型: /home/nvidia/microduck-course-run/converted/motrix_ball_balance_restored.opset11.onnx
开始生成 GPU 推理视频。


GPU 策略推理已启用；不展示具体显卡型号。
PASS-GPU-VIDEO: /home/nvidia/microduck-course-run/topic/03-Notebook/outputs/gpu_videos/ball_balance_gpu_latest.mp4
PASS-GPU-REPORT: /home/nvidia/microduck-course-run/topic/03-Notebook/outputs/gpu_videos/ball_balance_gpu_latest.json
2026-10-21 19:13:15.178620074 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
PASS-GPU-VIDEO: /home/nvidia/microduck-course-run/topic/03-Notebook/outputs/gpu_videos/ball_balance_gpu_latest.mp4
{'task': 'ball_balance', 'run_id': 'b60d8ee0-7a3d-4049-a195-87a7a6895832', 'generated_at_utc': '2026-10-21T11:13:28.475255+00:00', 'policy_validation': {'inference_calls': 900, 'real_observation_parity_checks': 32, 'max_abs_output_error': 2.682209014892578e-07, 'parity_atol': 0.0001, 'parity_rtol': 0.0001}, 'task_id': 'microduck-ball-balance', 'display_name': 'Motrix FastSAC ball balance', 'bac

## 6. RDK X5 BPU-in-the-loop 视频

本单元只展示新生成的板端闭环 MP4，不读取参考 GIF 或关键帧。Ubuntu 运行 MuJoCo 物理和 EGL 渲染；每一个控制步把任务观测发送到 RDK，RDK 用当前任务对应的 HBM 在 BPU 上计算 14 维动作，再返回给 Ubuntu。视频因此是“板端 BPU 产生动作、Ubuntu 负责仿真画面”的真实闭环。

运行前会自动重启 RDK 上的任务策略服务。切换 Notebook 时必须重新运行本单元，让板端 HBM 与当前任务匹配。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。

In [10]:
RUN_BPU = os.getenv("MICRODUCK_RUN_BPU", "0") == "1"
if not RUN_BPU:
    print("NOT-RUN-BPU: 板端推理未启用；设置 MICRODUCK_RUN_BPU=1 后连接 RDK 实际运行。")
else:
    import hashlib
    import uuid
    BPU_RUN_ID = str(uuid.uuid4())
    bpu_env = os.environ.copy()
    bpu_env['MICRODUCK_VALIDATION_RUN_ID'] = BPU_RUN_ID
    import sys

    BPU_TASK = 'ball_balance'
    BPU_LADDER_LOCAL_OPT_IN = os.getenv("MICRODUCK_RUN_LOCAL_LADDER_BPU", "0") == "1"
    BPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_BPU_VIDEO_STEPS", "1500" if BPU_TASK == "basketball" else "900" if BPU_TASK == "ball_balance" else "500"))
    BPU_VIDEO_WIDTH = int(os.getenv("MICRODUCK_BPU_VIDEO_WIDTH", "640"))
    BPU_VIDEO_HEIGHT = int(os.getenv("MICRODUCK_BPU_VIDEO_HEIGHT", "480"))
    BPU_CAMERA_DISTANCE = os.getenv("MICRODUCK_BPU_CAMERA_DISTANCE", "1.2")
    BPU_CAMERA_ELEVATION = os.getenv("MICRODUCK_BPU_CAMERA_ELEVATION", "-8")
    BPU_CAMERA_AZIMUTH = os.getenv("MICRODUCK_BPU_CAMERA_AZIMUTH", "90")
    BPU_PORT = int(os.getenv("MICRODUCK_BPU_PORT", "8765"))
    RDK_CONTROL_PORT = int(os.getenv("MICRODUCK_BPU_CONTROL_PORT", "8766"))
    RDK_HOST = os.getenv("RDK_HOST", "192.168.8.128")
    BPU_HBM_DEFAULTS = {
        "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ball_balance_restored_int16_normalized_split_expanded/model_output/ball_balance_restored_int16_normalized_split_expanded.bin",
        "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/basketball_recalibrated_int16/model_output/basketball_recalibrated_int16.bin",
        "walking": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
        "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
        "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/stilts25/model_output/stilts25.opset11.bin",
        "swing": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/swing/model_output/swing.opset11.bin",
        "ladder": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ladder/model_output/ladder.opset11.bin",
    }
    RDK_HBM_DEFAULTS = {
        "ball_balance": "/home/sunrise/microduck_policy/hbm/ball_balance.bin",
        "basketball": "/home/sunrise/microduck_policy/hbm/basketball.bin",
        "walking": "/home/sunrise/microduck_policy/hbm/walking.bin",
        "perturbation": "/home/sunrise/microduck_policy/hbm/walking.bin",
        "stilt": "/home/sunrise/microduck_policy/hbm/stilts25.bin",
        "swing": "/home/sunrise/microduck_policy/hbm/swing.bin",
        "ladder": "/home/sunrise/microduck_policy/hbm/ladder.bin",
    }
    BPU_HBM = Path(os.getenv("MICRODUCK_BPU_HBM", BPU_HBM_DEFAULTS.get(BPU_TASK, ""))) if BPU_TASK else None
    RDK_HBM = RDK_HBM_DEFAULTS.get(BPU_TASK, "") if BPU_TASK else ""
    RDK_SERVER_SCRIPT = os.getenv("RDK_BPU_SERVER_SCRIPT", "/home/sunrise/microduck_policy/rdk_bpu_policy_server.py")
    MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if BPU_TASK == "ball_balance" else None
    MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
    MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if BPU_TASK == "ball_balance" else None
    BPU_VIDEO_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_bpu_inloop_video.py" if BPU_TASK == "ball_balance" else "bpu_inloop_video.py")
    BPU_VIDEO_OUTPUT = OUTPUT_ROOT / "bpu_videos" / f"{BPU_TASK}_bpu_latest.mp4" if BPU_TASK else None
    ALLOW_CACHED_BPU = False
    BPU_LADDER_PAIR_SCRIPT = PLAYGROUND_ROOT / "scripts" / "bpu_official_ladder_pair_video.py"
    BPU_LADDER_CLIMBER_HBM = Path(os.getenv("MICRODUCK_BPU_LADDER_CLIMBER_HBM", "/home/ubuntu/workspaces/microduck_bpu_models/compiled/official_ladder_climber_feature_normalized_all_int16/model_output/official_ladder_climber_feature_normalized_all_int16.bin")).expanduser()
    BPU_LADDER_GETUP_HBM = Path(os.getenv("MICRODUCK_BPU_LADDER_GETUP_HBM", "/home/ubuntu/workspaces/microduck_bpu_models/compiled/official_ladder_getup_feature_combined_all_int16/model_output/official_ladder_getup_feature_combined_all_int16.bin")).expanduser()
    BPU_LADDER_PAIR_OUTPUT = OUTPUT_ROOT / "bpu_videos" / "ladder_official_pair_bpu_latest.mp4"

    def show_cached_bpu_video(path, reason):
        if not ALLOW_CACHED_BPU or path is None or not path.is_file():
            return False
        print("BPU-CACHED-VIDEO:", path)
        print("说明:", reason)
        report = path.with_suffix(".json")
        if report.is_file():
            print(json.loads(report.read_text(encoding="utf-8")))
        from IPython.display import Video, display
        display(Video(str(path), embed=True))
        return True

    def restart_rdk_policy_server():
        global BPU_BOARD_SHA
        if not BPU_TASK or not RDK_HBM:
            return False
        try:
            if BPU_HBM is not None and BPU_HBM.is_file():
                payload = BPU_HBM.read_bytes()
                request = {
                    "op": "upload_and_switch",
                    "task": BPU_TASK,
                    "size": len(payload),
                    "sha256": hashlib.sha256(payload).hexdigest(),
                }
            else:
                payload = b""
                request = {"op": "switch", "task": BPU_TASK}
            transform_path = BPU_HBM.with_suffix(".input_transform.json")
            if payload and transform_path.is_file():
                request["input_transform"] = json.loads(transform_path.read_text(encoding="utf-8"))
            with socket.create_connection((RDK_HOST, RDK_CONTROL_PORT), timeout=8) as control:
                control.sendall((json.dumps(request) + "\n").encode("utf-8"))
                if payload:
                    for offset in range(0, len(payload), 1024 * 1024):
                        control.sendall(payload[offset:offset + 1024 * 1024])
                response = json.loads(control.makefile("rb").readline().decode("utf-8"))
            if response.get("ok"):
                BPU_BOARD_SHA = response.get("sha256")
                if BPU_BOARD_SHA != hashlib.sha256(BPU_HBM.read_bytes()).hexdigest():
                    raise ValueError("板端模型 SHA 与所选 HBM 不一致。")
                action = "已上传并切换" if payload else "已切换"
                print("RDK 任务策略服务" + action + ":", BPU_TASK, "->", response.get("model"), "pid=", response.get("pid"))
                return True
            print("RDK BPU 控制服务拒绝切换:", response)
        except (OSError, ValueError, json.JSONDecodeError) as exc:
            print("RDK BPU 控制服务不可用，尝试 SSH:", exc)
        remote = "sunrise@" + RDK_HOST
        command = (
            "pkill -f '^python3 .*rdk_bpu_policy_server.py' >/dev/null 2>&1 || true; "
            "sleep 1; "
            "nohup python3 " + shlex.quote(RDK_SERVER_SCRIPT) +
            " --model " + shlex.quote(RDK_HBM) +
            " --port " + str(BPU_PORT) +
            " >/tmp/microduck_bpu_server.log 2>&1 </dev/null &"
        )
        result = subprocess.run(
            ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", remote, "bash -lc " + shlex.quote(command)],
            capture_output=True, text=True,
        )
        if result.returncode:
            print("RDK 策略服务启动失败:", result.stderr.strip() or result.stdout.strip())
            return False
        print("RDK 任务策略服务已切换:", BPU_TASK, "->", RDK_HBM)
        return True

    if BPU_TASK == "ladder":
        pair_ready = BPU_LADDER_PAIR_SCRIPT.is_file() and BPU_LADDER_CLIMBER_HBM.is_file() and BPU_LADDER_GETUP_HBM.is_file()
        with socket.create_connection((RDK_HOST, RDK_CONTROL_PORT), timeout=8) as control:
            control.sendall(b'{"op":"switch_pair"}\n')
            pair_status = json.loads(control.makefile("rb").readline())
        if not pair_status.get("ok"):
            raise RuntimeError(pair_status)
        pair_hashes = [pair_status["models"][role]["sha256"] for role in ("climber", "getup")]
        pair_cmd = [
            str(POLICY_PYTHON), str(BPU_LADDER_PAIR_SCRIPT), "--backend", "bpu",
            "--climber-hbm", str(BPU_LADDER_CLIMBER_HBM), "--getup-hbm", str(BPU_LADDER_GETUP_HBM),
            "--output", str(BPU_LADDER_PAIR_OUTPUT), "--seconds", os.getenv("MICRODUCK_LADDER_SECONDS", "24"),
            "--num-envs", "64", "--record-env", "17",
        ]
        print("BPU 官方双模型:", BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM)
        if not pair_ready:
            raise FileNotFoundError("缺少 climber/get-up HBM 或脚本。")
        else:
            BPU_LADDER_PAIR_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
            pair_result = subprocess.run(pair_cmd, cwd=PLAYGROUND_ROOT, env=bpu_env, capture_output=True, text=True, timeout=3600)
            print("\n".join((pair_result.stdout + "\n" + pair_result.stderr).splitlines()[-35:]))
            if pair_result.returncode != 0 or not BPU_LADDER_PAIR_OUTPUT.is_file():
                if not show_cached_bpu_video(BPU_LADDER_PAIR_OUTPUT, "本轮板端不可达，保留此前真实 RDK X5 BPU 双模型视频。"):
                    raise RuntimeError("官方爬梯 BPU 双模型视频生成失败。")
            else:
                print("PASS-BPU-VIDEO:", BPU_LADDER_PAIR_OUTPUT)
                pair_report = BPU_LADDER_PAIR_OUTPUT.with_suffix(".json")
                if pair_report.is_file():
                    print(verify_bpu_report(BPU_LADDER_PAIR_OUTPUT, BPU_RUN_ID, [BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM], pair_hashes))
                    print("PASS-FRESH-BPU: 本次双模型板端动作已驱动仿真并生成视频。")
                from IPython.display import Video, display
                display(Video(str(BPU_LADDER_PAIR_OUTPUT), embed=True))
    elif not BPU_TASK:
        print("该任务目前没有可验证的任务专属 HBM；不会用其他任务视频替代。")
    elif BPU_TASK == "ladder" and not BPU_LADDER_LOCAL_OPT_IN:
        print("已跳过本地 14 级 ladder HBM 视频：它不是社区 climber/get-up 方案。")
        print("只有明确设置 MICRODUCK_RUN_LOCAL_LADDER_BPU=1 才会运行本地实验，不得将结果标为社区复刻。")
    elif BPU_HBM is None or not BPU_HBM.is_file():
        raise FileNotFoundError(BPU_HBM)
    elif not BPU_VIDEO_SCRIPT.is_file():
        raise FileNotFoundError(BPU_VIDEO_SCRIPT)
    elif BPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
        print("找不到 MotrixLab Python 环境:", MOTRIX_PYTHON)
    elif BPU_TASK == "ball_balance" and (MOTRIX_RUNNER is None or not MOTRIX_RUNNER.is_file()):
        print("找不到 MotrixLab 启动器:", MOTRIX_RUNNER)
    elif restart_rdk_policy_server():
        BPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        if BPU_TASK == "ball_balance":
            video_cmd = [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(BPU_VIDEO_SCRIPT)]
        else:
            video_cmd = [str(POLICY_PYTHON), str(BPU_VIDEO_SCRIPT)]
        if BPU_TASK != "ball_balance":
            video_cmd += ["--task", BPU_TASK]
        video_cmd += [
            "--bpu-host", RDK_HOST, "--bpu-port", str(BPU_PORT),
            "--hbm", str(BPU_HBM), "--output", str(BPU_VIDEO_OUTPUT),
            "--steps", str(BPU_VIDEO_STEPS), "--width", str(BPU_VIDEO_WIDTH),
            "--height", str(BPU_VIDEO_HEIGHT),
        ]
        video_cmd += [
            "--camera-distance", BPU_CAMERA_DISTANCE,
            "--camera-elevation", BPU_CAMERA_ELEVATION,
            "--camera-azimuth", BPU_CAMERA_AZIMUTH,
        ]
        print("开始生成板端闭环视频:", " ".join(shlex.quote(x) for x in video_cmd))
        video_result = subprocess.run(
            video_cmd, cwd=PLAYGROUND_ROOT, env=bpu_env,
            capture_output=True, text=True,
        )
        print("\n".join((video_result.stdout + "\n" + video_result.stderr).splitlines()[-35:]))
        if video_result.returncode != 0 or not BPU_VIDEO_OUTPUT.is_file():
            raise RuntimeError("BPU-in-the-loop 视频生成失败，请先查看上面最后 35 行日志。")
        BPU_REPORT = BPU_VIDEO_OUTPUT.with_suffix(".json")
        print("PASS-BPU-VIDEO:", BPU_VIDEO_OUTPUT)
        if BPU_REPORT.is_file():
            print(verify_bpu_report(BPU_VIDEO_OUTPUT, BPU_RUN_ID, [BPU_HBM], [BPU_BOARD_SHA]))
            print("PASS-FRESH-BPU: 本次板端动作已驱动仿真并生成视频。")
        from IPython.display import Video, display
        display(Video(str(BPU_VIDEO_OUTPUT), embed=True))
    else:
        if not show_cached_bpu_video(BPU_VIDEO_OUTPUT, "板端不可达，展示历史视频。"):
            raise ConnectionError("RDK 板端推理未完成；请检查开发板网络与策略服务。")

    SIM_COMMAND = 'python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11'
    print("MuJoCo 回放命令模板:", SIM_COMMAND)

NOT-RUN-BPU: 板端推理未启用；设置 MICRODUCK_RUN_BPU=1 后连接 RDK 实际运行。


## 7. RDK X5 / BPU 审计

上一单元已经生成任务专属闭环视频。本单元只检查板端连通、HBM 元数据和服务日志；通用 MobileNet 样例只能证明系统运行时存在，不能代替任务视频。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。

In [11]:
RUN_BPU = os.getenv("MICRODUCK_RUN_BPU", "0") == "1"
if not RUN_BPU:
    print("NOT-RUN-BPU: 本轮只运行工作站推理。")
else:
    print("板端推理结果:")
    if BPU_TASK == "ladder":
        board_report = verify_bpu_report(BPU_LADDER_PAIR_OUTPUT, BPU_RUN_ID,
            [BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM], pair_hashes)
    else:
        board_report = verify_bpu_report(BPU_VIDEO_OUTPUT, BPU_RUN_ID, [BPU_HBM], [BPU_BOARD_SHA])
    print("PASS-FRESH-BPU: 模型 SHA、运行编号与实际控制步数已核对。")
    print("控制步数:", board_report["steps"])
    print("重置次数:", board_report.get("resets", board_report.get("terminated_or_truncated_resets", 0)))
    print("视频由本次任务策略驱动物理仿真生成。")

NOT-RUN-BPU: 本轮只运行工作站推理。


## 8. 阅读本次运行结果

先查看模型接口，再查看任务闭环与生成的视频，最后对照板端报告。

| 输出标记 | 对应流程 |
| --- | --- |
| `PASS-local-onnx` | 模型结构、输入输出形状和有限值检查通过；随后进入任务闭环 |
| `PASS-GPU-VIDEO` | GPU 策略连接物理仿真，完成本次视频与报告生成 |
| `PASS-BPU-VIDEO` | 当前连接 RDK，逐步取得对应 HBM 的动作，驱动工作站仿真并生成视频 |
| `BPU-CACHED-VIDEO` | 播放之前保存的板端演示；连接 RDK 后重跑板端闭环单元可生成新视频 |
| `PASS-rdk-bpu-runtime-sample` | 开发板的 MobileNet 样例完成推理，运行时检查通过 |
| `PASS-rdk-bpu` | 当前任务 HBM 的板端推理完成，输入输出与耗时保存在输出中 |

任务效果结合位移、速度、稳定时间、接触或登桌指标与视频一起阅读。
修改模型、命令或场景后重新运行对应闭环单元，保存自己的实验记录。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。